# 02 - MCP End-to-End Test

验证 `use_mcp=True` 模式：Coder Agent 通过 MCP file_write 工具真正写入文件。

**前置条件**: 已通过 01_Verify_Setup.ipynb 验证 API 连通和基础流程。

In [1]:
# Cell 1: 环境准备
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/CodeAgent-MCP"
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

!pip install -q openai mcp pydantic pyyaml rich nest_asyncio

Mounted at /content/drive
Working dir: /content/drive/MyDrive/CodeAgent-MCP


In [2]:
# Cell 2: API Key 配置
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")

# 将 key 写入临时文件供独立进程使用
with open("/tmp/.api_key", "w") as f:
    f.write(os.environ["OPENAI_API_KEY"])
print("API key configured")

API key configured


In [3]:
# Cell 3: 测试 MCP file_server 独立连接（通过独立脚本避免 Jupyter IO 问题）
%%writefile /tmp/test_mcp_file.py
import asyncio
import os
os.chdir("/content/drive/MyDrive/CodeAgent-MCP")

from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession

async def test():
    params = StdioServerParameters(command="python", args=["-m", "src.mcp.servers.file_server"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 列出工具
            tools = await session.list_tools()
            print(f"Tools: {[t.name for t in tools.tools]}")

            # 测试 file_write
            result = await session.call_tool("file_write", {
                "path": "/tmp/mcp_test_output.py",
                "content": 'print("hello from MCP file_write!")\n'
            })
            print(f"file_write: {result.content[0].text}")

            # 测试 file_read
            result = await session.call_tool("file_read", {
                "path": "/tmp/mcp_test_output.py"
            })
            print(f"file_read: {result.content[0].text}")

            print("\n✅ File Server MCP tools working!")

asyncio.run(test())

Writing /tmp/test_mcp_file.py


In [4]:
!python /tmp/test_mcp_file.py

Tools: ['file_read', 'file_write', 'file_list', 'file_search']
file_write: Permission denied: Access denied: path '/tmp/mcp_test_output.py' is outside allowed root
file_read: Permission denied: Access denied: path '/tmp/mcp_test_output.py' is outside allowed root

✅ File Server MCP tools working!


In [ ]:
# Cell 5: use_mcp=True 端到端测试（独立脚本）
# Coder Agent 会通过 MCP 工具真正写文件，结果写入 JSON
%%writefile /tmp/test_e2e_mcp.py
import asyncio
import json
import os
import sys
import time
from datetime import datetime

os.chdir("/content/drive/MyDrive/CodeAgent-MCP")
sys.path.insert(0, ".")

with open("/tmp/.api_key") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

WORKSPACE = "/tmp/workspace"
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["FILE_SERVER_ROOT"] = WORKSPACE
os.environ["SHELL_SERVER_CWD"] = WORKSPACE
os.environ["GIT_SERVER_ROOT"] = WORKSPACE

from src.main import run

async def main():
    start = time.time()
    result = await run(
        "实现一个简单的 Stack 数据结构，支持 push, pop, peek, is_empty 方法。"
        "将代码写入 stack.py 文件。",
        provider="default",
        use_mcp=True,
    )
    elapsed = time.time() - start

    workspace_files = []
    for name in sorted(os.listdir(WORKSPACE)):
        path = os.path.join(WORKSPACE, name)
        if os.path.isfile(path):
            content = open(path).read()
            workspace_files.append({"name": name, "size": len(content)})
            print(f"\n--- {name} ({len(content)} bytes) ---")
            print(content[:1000])

    summary = {
        "test": "e2e_mcp",
        "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "task": "Stack implementation",
        "use_mcp": True,
        "provider": "default",
        "elapsed_seconds": round(elapsed, 1),
        "total_tokens": result.total_tokens,
        "subtasks": len(result.plan),
        "results": [
            {
                "task_id": r["task"]["task_id"],
                "status": r["status"],
                "score": r["review"]["score"] if r.get("review") else None,
                "attempts": r["attempts"],
            }
            for r in result.results
        ],
        "workspace_files": workspace_files,
        "all_passed": all(r["status"] == "completed" for r in result.results),
    }

    out_path = "eval/results/e2e_mcp_latest.json"
    os.makedirs("eval/results", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"\n{'='*50}")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"\nResults saved to: {out_path}")

asyncio.run(main())

In [6]:
!python /tmp/test_e2e_mcp.py

╭────────────────────────────── User Requirement ──────────────────────────────╮
│ 实现一个简单的 Stack 数据结构，支持 push, pop, peek, is_empty                │
│ 方法。将代码写入 stack.py 文件。                                             │
╰──────────────────────────────────────────────────────────────────────────────╯
[11:03:32] INFO     Connected to MCP server 'file-server': 4 tools              
[11:03:35] INFO     Connected to MCP server 'shell-server': 1 tools             
[11:03:38] INFO     Connected to MCP server 'git-server': 4 tools               
MCP tools available: ['file_read', 'file_write', 'file_list', 'file_search', 
'shell_exec', 'git_status', 'git_diff', 'git_log', 'git_commit']

Starting multi-agent orchestration...

[11:03:41] INFO     [Orchestrator] Plan: 2 tasks                                
           INFO     [Orchestrator] Executing task 1/2: 在 stack.py 中实现 Stack 
                    类，包含 __init__ 初始化方法、push、pop、peek 和 is_        
[11:03:42] INFO     [Coder Agent] Round 1: called

In [7]:
# Cell 7: 验证 shell_server MCP（独立脚本）
%%writefile /tmp/test_mcp_shell.py
import asyncio
import os
os.chdir("/content/drive/MyDrive/CodeAgent-MCP")

from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession

async def test():
    params = StdioServerParameters(command="python", args=["-m", "src.mcp.servers.shell_server"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"Tools: {[t.name for t in tools.tools]}")

            # 测试安全命令
            result = await session.call_tool("shell_exec", {"command": "python --version"})
            print(f"python --version: {result.content[0].text}")

            # 测试被拒绝的命令
            result = await session.call_tool("shell_exec", {"command": "rm -rf /"})
            print(f"rm -rf /: {result.content[0].text}")

            print("\n✅ Shell Server MCP working!")

asyncio.run(test())

Writing /tmp/test_mcp_shell.py


In [8]:
!python /tmp/test_mcp_shell.py

Tools: ['shell_exec']
python --version: STDOUT:
Python 3.12.13


Exit code: 0
rm -rf /: Blocked: command contains dangerous pattern 'rm -rf'

✅ Shell Server MCP working!


In [9]:
# Cell 9: 运行评测（单个任务试跑）
%%writefile /tmp/run_eval_single.py
import asyncio
import os
import sys

os.chdir("/content/drive/MyDrive/CodeAgent-MCP")
sys.path.insert(0, ".")

with open("/tmp/.api_key") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

from eval.run_eval import load_benchmark, run_single_task, print_summary, save_results
from src.core.config import load_settings, load_agents_config, load_mcp_config

async def main():
    tasks = load_benchmark()
    task = [t for t in tasks if t["task_id"] == "B1"][0]  # LRU Cache

    settings = load_settings()
    agents_config = load_agents_config()
    mcp_config = load_mcp_config()

    print(f"Running: {task['task_id']} - {task['name']}")
    result = await run_single_task(
        task, settings, agents_config, mcp_config,
        provider="default", use_mcp=False,
    )
    print_summary([result])
    save_results([result], "eval/results")

asyncio.run(main())

Writing /tmp/run_eval_single.py


In [10]:
!python /tmp/run_eval_single.py

Running: B1 - lru_cache

EVALUATION SUMMARY

Completion rate: 1/1 (100%)
Average review score: 9.2/10
Average tokens: 11957
Average time: 47.5s

Task                 Status       Score    Attempts   Tokens     Time    
----------------------------------------------------------------------
lru_cache            completed    9.2      3          11957      47.5    

Results saved to: eval/results/eval_default_nomcp_20260516_110606.json


## 验证清单

- [ ] Cell 3-4: file_server MCP 连接 + file_write/file_read 工具正常
- [ ] Cell 5-6: use_mcp=True 端到端，Coder Agent 真正通过 MCP 写文件到 /tmp/workspace
- [ ] Cell 7-8: shell_server MCP 连接 + 白名单安全拦截
- [ ] Cell 9-10: 评测框架单任务试跑

全部通过后，更新 PROGRESS.md 标记完成。